# 18 - Agent State

State is the information the agent carries through a workflow.

A useful distinction:

```text
messages = conversation
state = broader working information
```

Typical state:
- messages
- user goal
- retrieved evidence
- plan
- tool results
- counters
- approval status
- intermediate decisions

## Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "")
if CHAT_MODEL:
    print("Configured model:", CHAT_MODEL)
else:
    print("Set OPENAI_CHAT_MODEL for live model examples.")

# %pip install -qU langgraph typing-extensions

## 1. Define typed state

Typed state makes agent workflows easier to reason about.

In [ ]:
from typing import TypedDict, List

class AgentState(TypedDict):
    goal: str
    steps: List[str]
    tool_calls: int
    status: str

state: AgentState = {
    'goal': 'Research RAG',
    'steps': [],
    'tool_calls': 0,
    'status': 'started',
}

print(state)

## 2. State updates

A node should generally do one job and return a small state update.

In [ ]:
def research_step(state: AgentState):
    return {
        'steps': state['steps'] + ['searched documentation'],
        'tool_calls': state['tool_calls'] + 1,
        'status': 'researching',
    }

state = {**state, **research_step(state)}
print(state)

## 3. StateGraph

LangGraph makes state transitions explicit.

In [ ]:
from langgraph.graph import StateGraph, START, END

class SimpleState(TypedDict):
    text: str
    result: str

def upper_node(state: SimpleState):
    return {'result': state['text'].upper()}

builder = StateGraph(SimpleState)
builder.add_node('upper', upper_node)
builder.add_edge(START, 'upper')
builder.add_edge('upper', END)

graph = builder.compile()

print(graph.invoke({'text': 'hello agent', 'result': ''}))

## 4. Conditional routing

The next node can depend on state.

Example:
```text
status == "needs_research" → research
status == "ready" → answer
```

In [ ]:
class RouteState(TypedDict):
    score: int
    route: str

def decide(state: RouteState):
    return {'route': 'verify' if state['score'] < 8 else 'finish'}

def route(state: RouteState):
    return state['route']

builder = StateGraph(RouteState)
builder.add_node('decide', decide)
builder.add_node('verify', lambda s: {'route': 'finish'})
builder.add_node('finish', lambda s: {'route': 'finish'})
builder.add_edge(START, 'decide')
builder.add_conditional_edges('decide', route)
builder.add_edge('verify', END)
builder.add_edge('finish', END)

graph = builder.compile()
print(graph.invoke({'score': 5, 'route': ''}))

## 5. State design rules

Keep state:
- small
- typed
- explicit
- safe to serialize
- easy to inspect

Avoid stuffing arbitrary objects into state when a stable ID/reference is enough.

## 6. Evaluation and practice

Test:
- every route
- state after every node
- invalid state
- repeated loops
- termination

Practice: build a state containing `goal`, `plan`, `evidence`, `tool_results`, and `final_answer`.

## Evaluation

Evaluation:
- state correctness
- route correctness
- state serialization
- isolation between threads
- termination

Improvement:
- typed schemas
- narrow node responsibilities
- explicit transitions
- state versioning for long-lived systems.

## References / current documentation


- LangChain reference: https://reference.langchain.com/python/langchain/overview
- LangGraph reference: https://reference.langchain.com/python/langgraph/overview
- LangGraph persistence: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/persistence.mdx
- LangGraph interrupts: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/interrupts.mdx
- LangChain human-in-the-loop: https://github.com/langchain-ai/docs/blob/main/src/oss/langchain/human-in-the-loop.mdx
- Ragas metrics: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/
- MCP Python SDK v2: https://py.sdk.modelcontextprotocol.io/
- MCP Python SDK getting started: https://py.sdk.modelcontextprotocol.io/get-started/